# BE-HPG — Phase 4b: Largest-connected-component post-processing

Re-evaluates the **already-trained** checkpoints (no retraining) with and without
largest-connected-component (LCC) post-processing, applied **identically to all three models**.
LCC keeps only the biggest predicted blob (valid for a single organ) and removes off-target
false positives that inflate HD95/ASD. Same fixed eval episodes as Phase 4 (seed 1234).

**Run:** GPU runtime; run top-to-bottom; upload `be-hpg-src.zip` at cell 3. ~10 min.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/be-hpg'
print('Drive:', DRIVE_ROOT)

In [ ]:
!pip -q install timm medpy

In [ ]:
import os, sys, shutil
FORCE_UPLOAD = True
zip_drive = f'{DRIVE_ROOT}/be-hpg-src.zip'
if FORCE_UPLOAD or not os.path.exists(zip_drive):
    from google.colab import files
    print('Upload be-hpg-src.zip:')
    up = files.upload()
    with open(zip_drive, 'wb') as f:
        f.write(up[list(up)[0]])
code_dir = '/content/be-hpg'
shutil.rmtree(code_dir, ignore_errors=True); os.makedirs(code_dir, exist_ok=True)
os.system(f'unzip -q -o "{zip_drive}" -d "{code_dir}"')
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)
for _m in [k for k in sys.modules if k == 'src' or k.startswith('src.')]:
    del sys.modules[_m]
import inspect
from src.metrics.postprocess import largest_connected_component
print('code ready | LCC available =', callable(largest_connected_component))

In [ ]:
import torch, json
import pandas as pd
from src.utils.config import load_config
from src.utils.seed import set_seed
from src.engine.build import build_model
from src.engine.eval import eval_episodes
from src.engine.checkpoint import load_checkpoint
from src.data.sampler import sampler_from_npz

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NPZ = f'{DRIVE_ROOT}/data/spleen/spleen.npz'
MANIFEST = f'{DRIVE_ROOT}/data/spleen/manifest.csv'
mdf = pd.read_csv(MANIFEST)
EVAL_EP, EPISODE_SEED = 150, 1234
MODELS = {'protonet': 'protonet.yaml', 'meta_reweight': 'meta_reweight.yaml', 'be_hpg': 'be_hpg.yaml'}
print('device:', DEVICE)

In [ ]:
rows = []
for name, yml in MODELS.items():
    cfg = load_config(f'{code_dir}/configs/{yml}')
    model = build_model(cfg)
    ckpt = f'{DRIVE_ROOT}/checkpoints/{name}_spleen.pt'
    assert os.path.exists(ckpt), f'missing checkpoint {ckpt} (run Phase 4 full first)'
    load_checkpoint(ckpt, model, map_location=DEVICE)
    out = {'model': name, 'by_shot': {}, 'by_shot_lcc': {}}
    for k in (1, 5):
        for pp, key in [(None, 'by_shot'), ('lcc', 'by_shot_lcc')]:
            ev = sampler_from_npz(NPZ, mdf, 'test', k_shots=[k],
                                  query_size=cfg['episode']['query_size'], seed=EPISODE_SEED)
            out[key][f'{k}shot'] = eval_episodes(model, ev, episodes=EVAL_EP, k_shot=k,
                                                 device=DEVICE, postprocess=pp)
        r = out['by_shot'][f'{k}shot']; rl = out['by_shot_lcc'][f'{k}shot']
        rows.append((name, k, r['dice'], rl['dice'], r['hd95'], rl['hd95'], r['asd'], rl['asd']))
    json.dump(out, open(f'{DRIVE_ROOT}/results/{name}_spleen_lcc.json', 'w'), indent=2)
    print('saved', f'{name}_spleen_lcc.json')

print('\n%-14s %3s | %-13s | %-15s | %-13s' % ('model', 'k', 'Dice raw/LCC', 'HD95 raw/LCC', 'ASD raw/LCC'))
for n, k, d, dl, h, hl, a, al in rows:
    print('%-14s %3d | %.3f / %.3f | %6.1f / %6.1f | %5.1f / %5.1f' % (n, k, d, dl, h, hl, a, al))

## Report back
Download the 3 `results/<model>_spleen_lcc.json` into the repo `results/` folder, and paste
the printed raw/LCC table. I'll use the LCC-postprocessed numbers (applied to all models) for
the main comparison and note the post-processing in the paper.